In [2]:
from pathlib import Path
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image, ImageFilter


ROOT = Path(os.getcwd()).resolve().parents[0]
WAND_B = ROOT / "wandb_export" / "runs"
OUT = ROOT / "poster" / "figures" / "affectivevision"


RUNS = {
    "ckplus_inception": "cvxvk4uc",
    "ckplus_mobilenet": "9szmvxl6",
    "ckplus_custom": "7l2tgrtc",
    "fer2013_mobilenet": "05lsoc3f",
    "fer2013_inception": "0mdo7vzo",
    "fer2013_custom": "jox665vn",
}

# These values mirror the LaTeX template: a0poster uses 108.82 cm of text width
# for A0 landscape; the poster uses four columns, 100 pt gaps, and 3 pt rules.
PT_TO_CM = 2.54 / 72.27
TEXTWIDTH_CM = 108.82
COLUMNSEP_CM = 100 * PT_TO_CM
COLUMNRULE_CM = 3 * PT_TO_CM
COLUMN_WIDTH_CM = (TEXTWIDTH_CM - 3 * COLUMNSEP_CM - 3 * COLUMNRULE_CM) / 4
COLUMN_WIDTH_IN = COLUMN_WIDTH_CM / 2.54

SANS_FONT = "Helvetica"
BODY_FONT_PT_ON_POSTER = 20.0


def history(run_id):
    return pd.read_csv(WAND_B / run_id / "history.csv")


def plot_sizes(source_width_in, target_width_fraction=1.0):
    target_width_in = COLUMN_WIDTH_IN * target_width_fraction
    scale = target_width_in / source_width_in
    # Matplotlib sizes are specified before LaTeX scales the imported PDF.
    # Divide the desired final A0-poster size by that scale so axes and ticks
    # land near the surrounding poster text after \includegraphics.
    base = BODY_FONT_PT_ON_POSTER / scale
    return {
        "base": base,
        "tick": base * 0.92,
        "axis": base,
        "title": base * 1.14,
        "legend": base * 0.88,
        "value": base * 0.95,
    }


def configure_matplotlib(source_width_in, target_width_fraction=1.0):
    sizes = plot_sizes(source_width_in, target_width_fraction)
    plt.rcParams.update(
        {
            "font.family": "sans-serif",
            "font.sans-serif": [SANS_FONT, "Arial", "DejaVu Sans"],
            "font.size": sizes["base"],
            "axes.titlesize": sizes["title"],
            "axes.labelsize": sizes["axis"],
            "xtick.labelsize": sizes["tick"],
            "ytick.labelsize": sizes["tick"],
            "legend.fontsize": sizes["legend"],
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
        }
    )
    return sizes


def style_axes(ax):
    ax.grid(True, color="#d5dde5", linewidth=0.55, alpha=0.8)
    for spine in ax.spines.values():
        spine.set_color("#9aa7b4")
        spine.set_linewidth(0.7)


def learning_curves(run_key, filename, title):
    df = history(RUNS[run_key]).dropna(subset=["epoch"])
    epochs = df["epoch"].astype(float)

    source_width = 7.2
    sizes = configure_matplotlib(source_width)
    fig, axes = plt.subplots(1, 2, figsize=(source_width, 2.8), constrained_layout=True)

    axes[0].plot(epochs, df["Train Loss"], color="#1f77b4", linewidth=1.8, label="train")
    axes[0].plot(epochs, df["Val Loss"], color="#d62728", linewidth=1.8, linestyle="--", label="validation")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    style_axes(axes[0])
    axes[0].legend(frameon=False, loc="upper right")

    axes[1].plot(epochs, df["Train Accuracy"], color="#1f77b4", linewidth=1.8, label="train")
    axes[1].plot(epochs, df["Val Accuracy"], color="#d62728", linewidth=1.8, linestyle="--", label="validation")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy")
    axes[1].set_ylim(0.35, 1.03)
    style_axes(axes[1])
    axes[1].legend(frameon=False, loc="lower right")

    fig.savefig(OUT / filename, bbox_inches="tight")
    plt.close(fig)


def comparison_chart():
    labels = [
        "Inc.\nCK+",
        "Mob.\nCK+",
        "CNN\nCK+",
        "Mob.\nFER",
        "Inc.\nFER",
        "CNN\nFER",
    ]
    keys = [
        "ckplus_inception",
        "ckplus_mobilenet",
        "ckplus_custom",
        "fer2013_mobilenet",
        "fer2013_inception",
        "fer2013_custom",
    ]
    values = []
    for key in keys:
        df = history(RUNS[key])
        values.append(float(df["Val Accuracy"].max()) * 100.0)

    source_width = 7.2
    sizes = configure_matplotlib(source_width)
    fig, ax = plt.subplots(figsize=(source_width, 2.55), constrained_layout=True)
    colors = ["#1f77b4", "#1f77b4", "#1f77b4", "#d97904", "#d97904", "#d97904"]
    bars = ax.bar(labels, values, color=colors, width=0.64)
    ax.set_ylabel("Best validation accuracy (%)")
    ax.set_ylim(55, 103)
    style_axes(ax)
    ax.grid(axis="x", visible=False)
    for bar, value in zip(bars, values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            value + 0.9,
            f"{value:.1f}",
            ha="center",
            va="bottom",
            fontsize=sizes["value"],
            fontweight="bold",
        )
    fig.savefig(OUT / "best_accuracy_comparison.pdf", bbox_inches="tight")
    plt.close(fig)


def ckplus_confusion():
    labels = ["anger", "contempt", "disgust", "fear", "happy", "sadness", "surprise"]
    support = np.array([34, 12, 29, 17, 39, 23, 43])
    matrix = np.diag(support)

    source_width = 4.6
    sizes = configure_matplotlib(source_width, target_width_fraction=0.78)
    fig, ax = plt.subplots(figsize=(source_width, 4.0), constrained_layout=True)
    image = ax.imshow(matrix, cmap="Blues")
    ax.set_xticks(range(len(labels)), labels=labels, rotation=35, ha="right", fontsize=sizes["tick"])
    ax.set_yticks(range(len(labels)), labels=labels, fontsize=sizes["tick"])
    ax.set_xlabel("Predicted label", fontsize=sizes["axis"])
    ax.set_ylabel("True label", fontsize=sizes["axis"])
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            value = int(matrix[i, j])
            if value:
                ax.text(j, i, str(value), ha="center", va="center", color="white", fontsize=sizes["tick"], fontweight="bold")
    cbar = fig.colorbar(image, ax=ax, fraction=0.045, pad=0.03)
    cbar.ax.tick_params(labelsize=sizes["tick"])
    fig.savefig(OUT / "ckplus_inception_confusion.pdf", bbox_inches="tight")
    plt.close(fig)


def copy_gradcam_faces():
    sources = {
        "gradcam_disgust_hi.png": "a9u2gawo/files/media/images/Grad-CAM Heatmaps_64_665f9cc79c07a2e880f3.png",
        "gradcam_sadness_hi.png": "a9u2gawo/files/media/images/Grad-CAM Heatmaps_64_a7363ddca6812dda49b8.png",
        "gradcam_contempt_hi.png": "a9u2gawo/files/media/images/Grad-CAM Heatmaps_64_47f87f2ca60e569cc1ea.png",
        "gradcam_surprise_hi.png": "a9u2gawo/files/media/images/Grad-CAM Heatmaps_64_10e350d743db22eade69.png",
    }
    for target, rel in sources.items():
        image = Image.open(WAND_B / rel).convert("RGB")
        image = image.resize((900, 900), Image.Resampling.LANCZOS)
        image = image.filter(ImageFilter.UnsharpMask(radius=1.2, percent=120, threshold=3))
        image.save(OUT / target, optimize=True)


def main():
    OUT.mkdir(parents=True, exist_ok=True)
    learning_curves("ckplus_inception", "ckplus_inception_learning.pdf", "CK+ InceptionV3-LSTM")
    learning_curves("fer2013_mobilenet", "fer2013_mobilenet_learning.pdf", "FER2013 MobileNetV2-LSTM")
    comparison_chart()
    ckplus_confusion()
    copy_gradcam_faces()


if __name__ == "__main__":
    main()


1 extra bytes in post.stringData array
'created' timestamp seems very low; regarding as unix timestamp
1 extra bytes in post.stringData array
'created' timestamp seems very low; regarding as unix timestamp
1 extra bytes in post.stringData array
'created' timestamp seems very low; regarding as unix timestamp
1 extra bytes in post.stringData array
'created' timestamp seems very low; regarding as unix timestamp
